In [4]:
import os
import json
import time
import gzip
import shutil
from typing import Dict, List, Optional, Tuple, Union
from urllib.parse import urlparse, unquote
from dotenv import load_dotenv, find_dotenv
from pathlib import Path
import requests

In [5]:
class TomTomTrafficDataCollector:
    """
    Thu thập dữ liệu lịch sử từ TomTom Traffic Stats - Area Analysis
    - Khu vực: Polygon hoặc MultiPolygon (có helper chuyển BBOX -> Polygon/MultiPolygon)
    - Khung giờ: 24 slot 15' (07:00–10:00 và 15:00–18:00), chỉ MONDAY–FRIDAY
    - Tự động ACCEPT nếu NEED_CONFIRMATION
    - Tải JSON/GeoJSON/SHAPE (ZIP), tự giải nén gzip cho JSON/GeoJSON
    """
    BASE = "https://api.tomtom.com/traffic/trafficstats"
    TIMEZONE = "Asia/Ho_Chi_Minh"

    def __init__(self, api_key: str):
        if not api_key:
            raise ValueError("Missing TOMTOM_TRAFFIC_STATS_API_KEY")

        # Tìm thư mục cha có tên "Urban-Traffic-Links" từ cwd trở lên
        cwd = Path(os.getcwd()).resolve()
        root = None
        if cwd.name.lower() == "urban-traffic-links":
            root = cwd
        else:
            for p in cwd.parents:
                if p.name.lower() == "urban-traffic-links":
                    root = p
                    break
        if root is None:
            root = cwd / "Urban-Traffic-Links"

        self.output_dir = root / "data" / "raw" / "tomtom_stats"
        self.output_dir.mkdir(parents=True, exist_ok=True)

        self.api_key = api_key
        print(f"💾 Output directory set to: {self.output_dir}")

    # ---------- Helpers ----------
    @staticmethod
    def bbox_to_polygon(bbox: List[float]) -> Dict:
        """
        Convert bbox [lat_min, lat_max, lon_min, lon_max] -> GeoJSON Polygon (EPSG:4326).
        GeoJSON dùng thứ tự [lon, lat].
        """
        lat_min, lat_max, lon_min, lon_max = bbox
        coords = [
            [lon_min, lat_min],
            [lon_max, lat_min],
            [lon_max, lat_max],
            [lon_min, lat_max],
            [lon_min, lat_min],
        ]
        return {"type": "Polygon", "coordinates": [coords]}

    @staticmethod
    def bboxes_to_multipolygon(bboxes: List[List[float]]) -> Dict:
        """
        Convert nhiều bbox -> GeoJSON MultiPolygon (EPSG:4326)
        """
        polys = []
        for bbox in bboxes:
            lat_min, lat_max, lon_min, lon_max = bbox
            ring = [
                [lon_min, lat_min],
                [lon_max, lat_min],
                [lon_max, lat_max],
                [lon_min, lat_max],
                [lon_min, lat_min],
            ]
            polys.append([ring])  # mỗi polygon = [linear_ring]
        return {"type": "MultiPolygon", "coordinates": polys}

    @staticmethod
    def _build_15min_slots(blocks: List[Tuple[str, str]]) -> List[Dict]:
        """
        Tạo danh sách timeSets 15' rời rạc (tối đa 24 theo giới hạn Area Analysis)
        Dùng full day names: MONDAY..FRIDAY (TomTom yêu cầu full-name)
        """
        def minutes(hhmm: str) -> int:
            h, m = map(int, hhmm.split(":"))
            return h * 60 + m

        def hhmm_from_minutes(total: int) -> str:
            h = total // 60
            m = total % 60
            return f"{h:02d}:{m:02d}"

        day_names = ["MON","TUE","WED","THU","FRI"]
        time_sets = []
        for start, end in blocks:
            s, e = minutes(start), minutes(end)
            cur = s
            while cur < e:
                slot_start = hhmm_from_minutes(cur)
                slot_end = hhmm_from_minutes(min(cur + 15, e))
                name = f"T{slot_start.replace(':','')}_{slot_end.replace(':','')}"
                time_sets.append({
                    "name": name,
                    "timeGroups": [{
                        "days": day_names,
                        "times": [f"{slot_start}-{slot_end}"]
                    }]
                })
                cur += 15
        return time_sets[:24]

    # -------------------- CREATE JOB --------------------
    def create_area_analysis_job(
        self,
        geometry: Dict,
        date_from: str,
        date_to: str,
        job_name: str = "HCMC Q1+Q7 15min Slots",
        frcs: Optional[List[Union[str,int]]] = None,
        probe_source: str = "ALL",
        average_sample_size_threshold: int = 0,
    ) -> Optional[str]:
        """
        geometry: GeoJSON Polygon/MultiPolygon (EPSG:4326)
        date_from/date_to: 'YYYY-MM-DD' (tối đa 1 năm)
        frcs: danh sách FRC (0..7) dạng str hoặc int; mặc định 0..5 (loại 6–7)
        """
        # Chỉ lấy các đường chính, loại FRC 6-7 (đúng ý bạn)
        if frcs is None:
            frcs = [0,1,2,3,4,5,6,7]
        # Chuẩn hoá về string như TomTom API thường ví dụ: ["0","1",...]
        frcs = [str(x) for x in frcs]

        # 24 slot 15' cho 07:00–10:00 và 15:00–18:00 (tổng đúng 24)
        time_sets = self._build_15min_slots([("07:00", "10:00"), ("15:00", "18:00")])

        body = {
            "jobName": job_name,
            "distanceUnit": "KILOMETERS",
            "network": {
                "name": "HCMC Q1+Q7",
                "geometry": geometry,
                "timeZoneId": self.TIMEZONE,
                "frcs": frcs,
                "probeSource": probe_source
            },
            "dateRange": {
                "name": "Weekdays",
                "from": date_from,
                "to": date_to,
                # "excludedDaysOfWeek": ["SATURDAY", "SUNDAY"]
            },
            "timeSets": time_sets,
            "averageSampleSizeThreshold": average_sample_size_threshold
        }

        url = f"{self.BASE}/areaanalysis/1?key={self.api_key}"
        try:
            resp = requests.post(url, headers={"Content-Type": "application/json"}, json=body, timeout=60)
            resp.raise_for_status()
            jr = resp.json()
            if jr.get("responseStatus") == "OK" and jr.get("jobId"):
                job_id = str(jr["jobId"])
                print(f"✅ Job created: {job_id}")
                return job_id
            print("❌ Create job failed:", jr)
            return None
        except requests.HTTPError as e:
            print("❌ HTTPError:", getattr(e.response, "status_code", "?"), getattr(e.response, "text", ""))
            return None
        except Exception as e:
            print("❌ Exception when creating job:", e)
            return None

    def _status_url(self, job_id: str) -> str:
        return f"{self.BASE}/status/1/{job_id}?key={self.api_key}"

    def check_job_status(self, job_id: str) -> dict:
        try:
            r = requests.get(self._status_url(job_id), timeout=30)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print("❌ Status check error:", e)
            return {}

    def _accept_job(self, job_id: str) -> bool:
        url = f"{self.BASE}/status/1/{job_id}/accept?key={self.api_key}"
        try:
            r = requests.post(url, timeout=30)
            r.raise_for_status()
            print("👍 Accepted job for processing.")
            return True
        except Exception as e:
            # r có thể không tồn tại nếu lỗi sớm → dùng thông báo chung
            print("❌ Accept failed:", e)
            return False

    def wait_for_job_completion(self, job_id: str, max_wait_minutes: int = 60, poll_seconds: int = 20) -> Optional[dict]:
        """
        Chờ tới khi DONE / ERROR / REJECTED / CANCELLED.
        Nếu NEED_CONFIRMATION -> tự động ACCEPT 1 lần.
        """
        print(f"⏳ Waiting for job {job_id} to complete...")
        start = time.time()
        max_wait = max_wait_minutes * 60
        accepted = False

        while True:
            if time.time() - start > max_wait:
                print(f"⏰ Timeout sau {max_wait_minutes} phút")
                return None

            st = self.check_job_status(job_id)
            state = st.get("jobState", "UNKNOWN")
            print(f"   Status: {state} (elapsed: {int(time.time() - start)}s)")

            if state == "DONE":
                print("✅ Job completed.")
                return st
            if state in ["ERROR", "REJECTED", "CANCELLED"]:
                print(f"❌ Job failed: {state}")
                return None
            if state == "NEED_CONFIRMATION" and not accepted:
                if not self._accept_job(job_id):
                    return None
                accepted = True

            time.sleep(poll_seconds)

    # -------------------- DOWNLOAD RESULTS --------------------
    @staticmethod
    def _maybe_gunzip_bytes(data: bytes) -> bytes:
        try:
            return gzip.decompress(data)
        except OSError:
            return data

    @staticmethod
    def _safe_filename_from_url(url: str) -> str:
        path = unquote(urlparse(url).path)
        return os.path.basename(path)

    def _download_stream(self, url: str, dst_path: str):
        with requests.get(url, stream=True, timeout=180) as r:
            r.raise_for_status()
            with open(dst_path, "wb") as f:
                shutil.copyfileobj(r.raw, f)
        return dst_path

    def download_results(self, job_id: str) -> Optional[dict]:
        status = self.check_job_status(job_id)
        if status.get("jobState") != "DONE":
            print(f"⚠️ Job chưa DONE: {status.get('jobState')}")
            return None

        urls = status.get("urls", [])
        if not urls:
            print("❌ Không có URL kết quả.")
            return None

        saved = {}
        for u in urls:
            fname = self._safe_filename_from_url(u)
            local = os.path.join(self.output_dir, f"{job_id}_{fname}")
            print("↓ Download:", fname)
            self._download_stream(u, local)

            lower = fname.lower()
            if lower.endswith(".json") or lower.endswith(".geojson"):
                with open(local, "rb") as f:
                    raw = f.read()
                unzipped = self._maybe_gunzip_bytes(raw)
                if unzipped != raw:
                    uz_path = local.replace(".json", "_unzipped.json").replace(".geojson", "_unzipped.geojson")
                    with open(uz_path, "wb") as fo:
                        fo.write(unzipped)
                    key = "json" if uz_path.endswith(".json") else "geojson"
                    saved[key] = uz_path
                    print("✅ Unzipped →", uz_path)
                else:
                    key = "json" if lower.endswith(".json") else "geojson"
                    saved[key] = local
            elif lower.endswith(".zip") or lower.endswith(".shp.zip") or lower.endswith(".shape"):
                saved["shape_zip"] = local
            else:
                saved[fname] = local

        status_path = os.path.join(self.output_dir, f"{job_id}_status.json")
        with open(status_path, "w", encoding="utf-8") as f:
            json.dump(status, f, indent=2, ensure_ascii=False)
        saved["status"] = status_path

        print("📁 Saved files:", json.dumps(saved, indent=2, ensure_ascii=False))
        return saved

In [8]:
import os
load_dotenv(r"C:\AI\Specialized_Project\Urban-Traffic-Links\.env", override=True)
API_KEY = os.getenv("TOMTOM_TRAFFIC_STATS_API_KEY")
print("API_KEY =", API_KEY)         # phải thấy 5CVEKZR8ct2T10YLNStFEJuBWkYWktib
print("TYPE  =", type(API_KEY))     # phải là <class 'str'>
collector = TomTomTrafficDataCollector(API_KEY)

# BBOX trung tâm (Q1,3,4,5,10): [lat_min, lat_max, lon_min, lon_max]
Q1_BBOX = [10.7535894, 10.7969493, 106.6816889, 106.7151954]
geometry = TomTomTrafficDataCollector.bbox_to_polygon(Q1_BBOX)

# Ví dụ tạo job cho tháng 08/2024 (chỉ weekdays, 07-10 & 15-18, FRC 0..5)
# job_id = collector.create_area_analysis_job(
#     geometry=geometry,
#     date_from="2024-08-01",
#     date_to="2024-08-31",
#     job_name="HCMC CBD Weekday 07-10 & 15-18 (FRC0-5)"
# )

# Nếu đã có job_id có sẵn:
# job_id = collector.create_area_analysis_job(
#     geometry=geometry,
#     date_from="2024-08-01",
#     date_to="2024-08-31",
#     job_name="HCMC CBD Weekday 07-10 & 15-18 (FRC0-7)"
# )
job_id = "8285492"
print("New job_id =", job_id)
if job_id:
    status = collector.wait_for_job_completion(job_id, max_wait_minutes=60, poll_seconds=20)
    if status and status.get("jobState") == "DONE":
        files = collector.download_results(job_id)
    else:
        print("⚠️ Job chưa hoàn tất hoặc thất bại.")


API_KEY = 5CVEKZR8ct2T10YLNStFEJuBWkYWktib
TYPE  = <class 'str'>
💾 Output directory set to: C:\AI\Specialized_Project\Urban-Traffic-Links\data\raw\tomtom_stats
New job_id = 8285492
⏳ Waiting for job 8285492 to complete...
❌ Status check error: HTTPSConnectionPool(host='api.tomtom.com', port=443): Max retries exceeded with url: /traffic/trafficstats/status/1/8285492?key=5CVEKZR8ct2T10YLNStFEJuBWkYWktib (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1017)')))
   Status: UNKNOWN (elapsed: 16s)
❌ Status check error: HTTPSConnectionPool(host='api.tomtom.com', port=443): Max retries exceeded with url: /traffic/trafficstats/status/1/8285492?key=5CVEKZR8ct2T10YLNStFEJuBWkYWktib (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1017)')))
   Status: UNKNOWN (elapsed: 54s)
❌ Status check error: HTTPSConnectionPool(host='api.tomtom.com', port=443): Max retries 

In [ ]:
import os, pathlib
print("CWD =", os.getcwd())
print("OUTPUT_DIR (abs) =", pathlib.Path("./data/raw/tomtom_stats").resolve())
print("Files =", list(pathlib.Path("./data/raw/tomtom_stats").glob("*")))
